# Kaggle · Damage-focused evaluation (about 30 minutes)

Whole-tile SSIM/PSNR reward copying the before image, because most of a disaster scene does
not change. This scores the same held-out Kerala events on **where** damage is generated:
pixel-level precision / recall / F1 / IoU of the generated change mask against the real one,
plus SSIM/PSNR on the changed tiles only. Results go to `outputs/metrics_damage.json`.
Settings: Accelerator **GPU T4 x2**, Internet **On**, then Save Version → Save & Run All.

In [1]:
# Shared Google Drive links ("Anyone with the link"): the tiles and both checkpoint folders.
TILES_URL = "https://drive.google.com/file/d/1WC1wOosic5MNSI44HxkPHnxxwbgibxtZ/view?usp=sharing"
GAN_FOLDER_URL = "https://drive.google.com/drive/folders/1nr4aG_OQme16UpHARmxtpOm__y6qRID3"
DIFFUSION_FOLDER_URL = "https://drive.google.com/drive/folders/1yBvYIRXWNecgHBQ7v-KDFZnLq_Et-n_I"
REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/kaggle/working/kerala-hazard-vision-ai"

!git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q ".[sd]" gdown

/kaggle/working/kerala-hazard-vision-ai
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import zipfile
from pathlib import Path

import gdown

# Inputs go to /tmp: only /kaggle/working is saved as output.
gdown.download(TILES_URL, "/tmp/tiles.zip", fuzzy=True, quiet=True)
with zipfile.ZipFile("/tmp/tiles.zip") as archive:
    archive.extractall("/tmp/tiles")
for url, folder in ((GAN_FOLDER_URL, "/tmp/checkpoints"),
                    (DIFFUSION_FOLDER_URL, "/tmp/checkpoints_diffusion")):
    gdown.download_folder(url, output=folder, quiet=True)
print(sorted(p.name for p in Path("/tmp/checkpoints").iterdir()),
      sorted(p.name for p in Path("/tmp/checkpoints_diffusion").iterdir()))

['best_g.pt', 'config.json', 'history.csv', 'last.pt', 'stats.json'] ['best_ema.pt', 'config.json', 'history.csv', 'losses_diffusion.png', 'stats.json']


In [3]:
import pandas as pd
import torch

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.data.dataset import (
    TileDataset,
    list_tiles,
    load_shared_stats,
    to_reflectance,
)
from kerala_hazard_vision_ai.evaluate import (
    diffusion_predict,
    evaluate_damage,
    gan_predict,
    identity_predict,
    rgb_uint8,
    save_metrics,
)
from kerala_hazard_vision_ai.models import load_generator
from kerala_hazard_vision_ai.train_diffusion import load_diffusion

cfg = load_config("configs/default.yaml")
TILE_DIR = Path("/tmp/tiles")
OUT_DIR = Path("/kaggle/working/outputs")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
GAN_DIR, DIFFUSION_DIR = Path("/tmp/checkpoints"), Path("/tmp/checkpoints_diffusion")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fails loudly if the two models were trained on different tiles (e.g. after re-running 01).
stats = load_shared_stats(GAN_DIR, DIFFUSION_DIR)
test_events = [e for e in cfg.events if e.name in cfg.data.test_events]
test_sets = {e.name: TileDataset(list_tiles(TILE_DIR, [e.name]), stats, augment=False)
             for e in test_events}

generator = load_generator(GAN_DIR / "best_g.pt", cfg.model, device)
diffusion = load_diffusion(DIFFUSION_DIR / "best_ema.pt", cfg, device)
predictors = {
    "pix2pix": gan_predict(generator, device),
    "diffusion": diffusion_predict(diffusion, device, cfg.diffusion.sample_steps, cfg.train.seed),
}


def to_rgb(scaled: torch.Tensor) -> torch.Tensor:
    return rgb_uint8(to_reflectance(scaled, stats), cfg.eval.rgb_max)


def stack_tiles(ds: TileDataset, indices) -> tuple[torch.Tensor, torch.Tensor]:
    xs, ys = zip(*(ds[i] for i in indices), strict=True)
    return torch.stack(xs), torch.stack(ys)


for event in test_events:
    print(f"Test event: {event.name} ({event.hazard}), {len(test_sets[event.name])} tiles")

Test event: kuttanad_2018 (flood), 40 tiles
Test event: pettimudi_2020 (landslide), 256 tiles


## Damage scores per model and test event

In [4]:
predictors["identity"] = identity_predict(stats)
damage = {}
for event in test_events:
    damage[event.name] = {name: evaluate_damage(predict, test_sets[event.name], cfg)
                          for name, predict in predictors.items()}
save_metrics(damage, OUT_DIR / "metrics_damage.json")
pd.DataFrame([
    {"event": event, "model": model, **scores["damage"],
     **{f"changed_{k}": v for k, v in scores["changed_tiles"].items()}}
    for event, models in damage.items() for model, scores in models.items()
]).round(3)

,event,model,precision,recall,f1,iou,changed_n_tiles,changed_ssim,changed_psnr
0,kuttanad_2018,pix2pix,0.739,0.181,0.291,0.170,38,0.623,17.190
1,kuttanad_2018,diffusion,0.000,0.000,0.000,0.000,38,0.672,14.750
2,kuttanad_2018,identity,0.000,0.000,0.000,0.000,38,0.798,21.881
3,pettimudi_2020,pix2pix,0.013,0.000,0.000,0.000,0,NaN,NaN
4,pettimudi_2020,diffusion,0.002,0.138,0.004,0.002,0,NaN,NaN
5,pettimudi_2020,identity,0.696,0.001,0.002,0.001,0,NaN,NaN
